<a href="https://colab.research.google.com/github/FarukTeker/311ComputerArchitecture/blob/main/colab_session2_B4_M5_M6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Oturum 2 — B4: MiniGPT-4 + M5: LLaVA-Med + M6: Med-Flamingo

**Tahmini süre:** ~5.5 saat (A100)

### Başlamadan önce (ZORUNLU):
1. `Runtime → Change runtime type → A100 GPU` seç
2. Sol taraf 🔑 simgesi → `GITHUB_TOKEN` toggle'ını **aç**
3. Hücreleri **sırayla** çalıştır, hiçbirini atlama

### Kesinti olursa:
- Hücre 1-5'i tekrar çalıştır (setup)
- Sadece kesilen modelin hücresini `--resume` ile çalıştır
- Hücre 9 ile push et

In [2]:
# ── HÜCRE 1: Drive + GPU ──────────────────────────────────────────
from google.colab import drive, userdata
import os

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/MedVQA_Results'
os.makedirs(DRIVE_DIR, exist_ok=True)

# HF_TOKEN — LLaVA-Med gated modeli için (varsa kullan)
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN ayarlandı.')
except Exception:
    print('HF_TOKEN bulunamadı — public fallback model kullanılacak.')

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f'Drive hazır: {DRIVE_DIR}')

Mounted at /content/drive
HF_TOKEN bulunamadı — public fallback model kullanılacak.
NVIDIA A100-SXM4-40GB, 40960 MiB
Drive hazır: /content/drive/MyDrive/MedVQA_Results


In [3]:
# ── HÜCRE 2: Repo klonla (token ile) ─────────────────────────────
from google.colab import userdata
import subprocess, os

TOKEN = userdata.get('GITHUB_TOKEN')

subprocess.run(
    ['git', 'clone',
     f'https://{TOKEN}@github.com/FarukTeker/Multimodal_Biomedical_QA.git'],
    check=True
)
os.chdir('/content/Multimodal_Biomedical_QA')
print('Repo hazır:', os.listdir('.'))

Repo hazır: ['Kvasir-VQA-x1_2506.09958.pdf', 'multimodal_biomedical_qa_dataset_taramasi.txt', 'PMC-VQA_2305.10415.pdf', 'RadImageNet-VQA_2512.17396.pdf', 'experiments', 'WorldMedQA-V_2410.12722.pdf', 'ROCOv2_s41597-024-03496-6.pdf', '.gitignore', 'results', 'SMMILE_2506.21355.pdf', 'notebooks', 'BIOMEDICA_2501.07171.pdf', 'requirements.txt', 'MedTrinity-25M_2408.02900.pdf', 'scripts', 'ROADMAP.md', '.git', 'PathVQA_2003.10286.pdf', 'src', 'README.md', 'report', 'VQA-RAD_1809.10082.pdf']


In [4]:
# ── HÜCRE 3: Bağımlılıkları yükle ───────────────────────────────
import subprocess
# Ana bağımlılıklar
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'],
               cwd='/content/Multimodal_Biomedical_QA')
# Med-Flamingo için open-flamingo
subprocess.run(['pip', 'install', '-q', 'open-flamingo'],
               cwd='/content/Multimodal_Biomedical_QA')
print('Kurulum tamam.')

Kurulum tamam.


In [5]:
# ── HÜCRE 4: PMC-VQA görüntülerini indir ────────────────────────
# Drive'da yedek varsa oradan kopyalar (~2 dk)
# Yoksa HuggingFace'den indirir (~8 dk)
import os, shutil
from huggingface_hub import hf_hub_download
import zipfile

IMGS_DIR   = '/content/pmc_vqa_data/images'
DRIVE_IMGS = f'{DRIVE_DIR}/pmc_vqa_images'

os.makedirs(IMGS_DIR, exist_ok=True)

# Görüntüler zaten indirilmişse atla
existing = sum(len(os.listdir(os.path.join(IMGS_DIR, d)))
               for d in os.listdir(IMGS_DIR)
               if os.path.isdir(os.path.join(IMGS_DIR, d)))

if existing > 100000:
    print(f'Görüntüler zaten mevcut: {existing:,} dosya — atlanıyor.')
elif os.path.exists(DRIVE_IMGS) and os.path.isdir(DRIVE_IMGS):
    sub = sum(len(os.listdir(os.path.join(DRIVE_IMGS, d)))
              for d in os.listdir(DRIVE_IMGS)
              if os.path.isdir(os.path.join(DRIVE_IMGS, d)))
    if sub > 100000:
        print(f'Drive\'dan kopyalanıyor ({sub:,} dosya)...')
        shutil.copytree(DRIVE_IMGS, IMGS_DIR, dirs_exist_ok=True)
        print('Kopyalandı.')
    else:
        print('Drive\'da görüntü yok, HuggingFace\'den indiriliyor...')
else:
    for fname in ['images.zip', 'images_2.zip']:
        print(f'{fname} indiriliyor...')
        zpath = hf_hub_download(
            repo_id='xmcmic/PMC-VQA', filename=fname,
            repo_type='dataset', local_dir='/content/pmc_vqa_data'
        )
        print('Çıkartılıyor...')
        with zipfile.ZipFile(zpath, 'r') as z:
            z.extractall(IMGS_DIR)
        os.remove(zpath)
        print('Tamam.')

total = sum(len(os.listdir(os.path.join(IMGS_DIR, d)))
            for d in os.listdir(IMGS_DIR)
            if os.path.isdir(os.path.join(IMGS_DIR, d)))
print(f'\nToplam görüntü: {total:,}')

Drive'da görüntü yok, HuggingFace'den indiriliyor...

Toplam görüntü: 0


In [6]:
# ── HÜCRE 5: Dataset testi ──────────────────────────────────────
import sys
sys.path.insert(0, '/content/Multimodal_Biomedical_QA')
from src.data.dataset_loader import PMCVQADataset

ds = PMCVQADataset(split='test', max_samples=3,
                   images_dir='/content/pmc_vqa_data/images')
item = ds[0]
print('Soru  :', item['question'])
print('Cevap :', item['answer'])
print('Şık A :', item['choice_a'])
print('Görüntü:', item['image'].size)
print('\nDataset hazır, modellere geçiliyor!')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


test_clean.csv: 0.00B [00:00, ?B/s]

Generating test split: 0 examples [00:00, ? examples/s]

PMC-VQA loaded: 3 examples from test_clean.csv
Soru  : What is the name of the medical imaging technique used in this case?
Cevap : Magnetic resonance imaging
Şık A :  A:X-ray 
Görüntü: (224, 224)

Dataset hazır, modellere geçiliyor!


In [7]:
# ── HÜCRE 6: B4 — MiniGPT-4 (~90 dk) ───────────────────────────
import subprocess
r = subprocess.run([
    'python', 'scripts/run_baseline.py',
    '--model', 'minigpt4',
    '--dataset', 'pmc_vqa',
    '--resume',
    '--drive_dir', DRIVE_DIR
], cwd='/content/Multimodal_Biomedical_QA')
print('B4 MiniGPT-4 tamamlandı.' if r.returncode == 0 else 'B4 HATA!')

B4 MiniGPT-4 tamamlandı.


In [8]:
# ── HÜCRE 7: M5 — LLaVA-Med (~75 dk) ───────────────────────────
import subprocess
r = subprocess.run([
    'python', 'scripts/run_baseline.py',
    '--model', 'llava_med',
    '--dataset', 'pmc_vqa',
    '--resume',
    '--drive_dir', DRIVE_DIR
], cwd='/content/Multimodal_Biomedical_QA')
print('M5 LLaVA-Med tamamlandı.' if r.returncode == 0 else 'M5 HATA!')

M5 LLaVA-Med tamamlandı.


In [9]:
# ── HÜCRE 8: M6 — Med-Flamingo (~180 dk) ────────────────────────
# Not: Med-Flamingo OPT-6.7B kullanır (~14GB VRAM).
# Yeterli VRAM yoksa hata verebilir — o zaman bu hücreyi atla.
import subprocess
r = subprocess.run([
    'python', 'scripts/run_baseline.py',
    '--model', 'med_flamingo',
    '--dataset', 'pmc_vqa',
    '--resume',
    '--drive_dir', DRIVE_DIR
], cwd='/content/Multimodal_Biomedical_QA')
print('M6 Med-Flamingo tamamlandı.' if r.returncode == 0 else 'M6 HATA!')

M6 Med-Flamingo tamamlandı.


In [10]:
# ── HÜCRE 8.5: Sonuçları doğrula — push'tan ÖNCE çalıştır ───────
import json, os

expected = ['minigpt4', 'llava_med', 'med_flamingo']
found = {}

print('Drive sonuç kontrolü:\n')
for f in sorted(os.listdir(DRIVE_DIR)):
    if f.startswith('.') or not f.endswith('.json'):
        continue
    # Hangi modele ait?
    model = next((m for m in expected if f.startswith(m)), None)
    if model is None:
        continue
    try:
        with open(f'{DRIVE_DIR}/{f}') as fp:
            d = json.load(fp)
        n   = d.get('n_samples', '?')
        acc = d.get('metrics', {}).get('closed', {}).get('accuracy', None)
        acc_str = f'{acc:.3f}' if acc is not None else '?'
        print(f'  OK     {f}  (n={n}, acc={acc_str})')
        # En son dosyayı tut (birden fazla çalıştırma varsa)
        found[model] = {'file': f, 'n': n, 'acc': acc}
    except Exception as e:
        print(f'  BOZUK  {f}: {e}')

missing = [m for m in expected if m not in found]

print()
if missing:
    print(f'EKSIK / HATA: {missing}')
    print('Bu modellerin hücresini tekrar çalıştır, sonra bu hücreyi tekrar çalıştır.')
    raise SystemExit('Doğrulama başarısız — push yapılmıyor.')
else:
    print('Tüm modeller tamamlandı.')
    for m, v in found.items():
        print(f'  {m}: n={v["n"]}, acc={v["acc"]:.3f}')
    print('\nHücre 9 ile push edebilirsin.')

Drive sonuç kontrolü:

  OK     llava_med_pmc_vqa_20260417_1021.json  (n=2000, acc=0.085)
  OK     llava_med_pmc_vqa_20260417_1039.json  (n=2000, acc=0.085)
  OK     llava_med_pmc_vqa_20260417_1040_corrected.json  (n=2000, acc=0.297)
  OK     llava_med_pmc_vqa_20260417_1042_corrected.json  (n=2000, acc=0.297)
  OK     llava_med_pmc_vqa_20260508_1235.json  (n=2000, acc=0.297)
  OK     llava_med_pmc_vqa_20260508_1312.json  (n=2000, acc=0.297)
  OK     med_flamingo_pmc_vqa_20260417_1641.json  (n=2000, acc=0.062)
  OK     med_flamingo_pmc_vqa_20260508_1330.json  (n=2000, acc=0.062)
  OK     minigpt4_pmc_vqa_20260417_0951.json  (n=2000, acc=0.089)
  OK     minigpt4_pmc_vqa_20260417_1040_corrected.json  (n=2000, acc=0.230)
  OK     minigpt4_pmc_vqa_20260417_1042_corrected.json  (n=2000, acc=0.230)
  OK     minigpt4_pmc_vqa_20260508_1227.json  (n=2000, acc=0.230)
  OK     minigpt4_pmc_vqa_20260508_1303.json  (n=2000, acc=0.230)

Tüm modeller tamamlandı.
  llava_med: n=2000, acc=0.297
  med_fl

In [11]:
# ── HÜCRE 9: Sonuçlar + GitHub push ─────────────────────────────
import os, shutil, subprocess
from google.colab import userdata

cwd = '/content/Multimodal_Biomedical_QA'
tables_dir = f'{cwd}/results/tables'
os.makedirs(tables_dir, exist_ok=True)

# Drive'dan sadece gerçek sonuç dosyalarını kopyala (checkpoint hariç)
print('Drive sonuçları kopyalanıyor:')
copied = []
for f in sorted(os.listdir(DRIVE_DIR)):
    if f.startswith('.'):          # .ckpt_* dosyalarını atla
        continue
    if not (f.endswith('.json') or f.endswith('.csv')):
        continue
    src = f'{DRIVE_DIR}/{f}'
    dst = f'{tables_dir}/{f}'
    shutil.copy2(src, dst)
    copied.append(f)
    print(f'  {f}')

print(f'\nToplam {len(copied)} dosya kopyalandı.')

# GitHub'a push
TOKEN = userdata.get('GITHUB_TOKEN')
subprocess.run(['git', 'config', 'user.email', 'megafaruk2012@gmail.com'], cwd=cwd)
subprocess.run(['git', 'config', 'user.name', 'Omer Faruk Teker'], cwd=cwd)
subprocess.run(['git', 'remote', 'set-url', 'origin',
    f'https://{TOKEN}@github.com/FarukTeker/Multimodal_Biomedical_QA.git'], cwd=cwd)

# Önce pull (çakışma olmasın)
subprocess.run(['git', 'pull', '--rebase', 'origin', 'main'], cwd=cwd)

# Sonuç dosyalarını ekle
subprocess.run(['git', 'add', 'results/tables/'], cwd=cwd)

# Değişiklik var mı kontrol et
status = subprocess.run(['git', 'status', '--porcelain'], cwd=cwd,
                        capture_output=True, text=True)
if status.stdout.strip():
    subprocess.run(['git', 'commit', '-m', 'Oturum 2: B4-M6 sonuc JSON dosyalari'], cwd=cwd)
    r = subprocess.run(['git', 'push'], cwd=cwd)
    if r.returncode == 0:
        print('GitHub\'a kaydedildi!')
    else:
        print('Push hatasi!')
else:
    print('Yeni sonuç dosyası yok, push atlandı.')

Drive sonuçları kopyalanıyor:
  blip2_pmc_vqa_20260416_1413.json
  blip2_pmc_vqa_20260416_1621.json
  blip2_pmc_vqa_20260416_1824.json
  blip2_pmc_vqa_20260416_1835.json
  blip2_pmc_vqa_20260417_0743.json
  blip2_pmc_vqa_20260417_0805.json
  blip2_pmc_vqa_20260417_0951.json
  blip2_pmc_vqa_20260417_1040_corrected.json
  blip2_pmc_vqa_20260417_1042_corrected.json
  blip2_pmc_vqa_20260506_2004.json
  blip2_pmc_vqa_20260507_1537.json
  blip2_pmc_vqa_20260508_0814.json
  instructblip_pmc_vqa_20260416_1634.json
  instructblip_pmc_vqa_20260416_1844.json
  instructblip_pmc_vqa_20260417_0752.json
  instructblip_pmc_vqa_20260417_0801.json
  instructblip_pmc_vqa_20260417_0959.json
  instructblip_pmc_vqa_20260417_1040_corrected.json
  instructblip_pmc_vqa_20260417_1042_corrected.json
  instructblip_pmc_vqa_20260508_0823.json
  llava_med_pmc_vqa_20260417_1021.json
  llava_med_pmc_vqa_20260417_1039.json
  llava_med_pmc_vqa_20260417_1040_corrected.json
  llava_med_pmc_vqa_20260417_1042_corrected.jso